I Think the best lr/epoch combo would be 0.001 and 500

I want to test different lr schedulers

In [1]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial
import numpy as np


# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from model.functions import pinball_loss, rmse, train, get_test_loss, pinball_loss_tensor, pinball_loss_sum, normalize_batch, compute_mean_std

In [3]:
class thread_net(nn.Module):

    @staticmethod
    def _mlp(sizes: list[int]) -> nn.Sequential:
        """Linear layers with ReLU between; no ReLU after the last linear."""
        if len(sizes) < 2:
            raise ValueError("layer_list needs at least [input_dim, output_dim]")
        layers: list[nn.Module] = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        return nn.Sequential(*layers)

    def __init__(self, layer_list=[12, 40, 40, 40, 1], num_nets=500):
        super().__init__()
        self.layer_list = list(layer_list)
        self.nets = nn.ModuleList(
            [self._mlp(self.layer_list) for _ in range(num_nets)]
        )
        self.num_nets = num_nets

    def forward(self, x):
        outs = [net(x[:, i::self.num_nets]) for i, net in enumerate(self.nets)]
        return torch.cat(outs, dim=1)

In [4]:
all_specs = [
    "sales",
    "7_day_rolling_ema",	
    "30_day_rolling_ema",
    "90_day_rolling_ema",
    "30_day_rolling_min",
    "5_day_lag",
    "6_day_lag",
    "7_day_lag",
    "dif_180_day",
]

In [5]:
# Get data for spec
train_loader, val_loader, test_loader = create_dataloader(
    batch_size=8, 
    test_batch_size=1,
    pin_memory=False,
    specs=all_specs,
    combine_items=True,
    combine_stores=True
    )

In [6]:
# Parameters
h_cost = 1
l_cost = 3

num_nets = train_loader.dataset.y.shape[1]
in_per_net = train_loader.dataset.x.shape[1] // num_nets
layer_list = [in_per_net, 40, 40, 40, 1]  

In [ ]:
epochs = 1000
lrs = [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001, 0.00005, 0.00001]

In [8]:
mean_losses = []

for lr in lrs:

    net = thread_net(layer_list=layer_list, num_nets=num_nets)

    loss = partial(pinball_loss_sum, h_cost=h_cost, l_cost=l_cost)
    loss_tensor = partial(pinball_loss_tensor, h_cost=h_cost, l_cost=l_cost)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    mean, std = compute_mean_std(train_loader, device="cpu")
    normalize_func = partial(normalize_batch, mean=mean, std=std)

    _, _ = train(net, optimizer, loss, train_loader, val_loader, epochs=epochs, eval_interval=None, device="cpu", use_tqdm=False, normalize_func=normalize_func)

    test_loss = get_test_loss(net, test_loader, loss, "cpu")
    average_test_loss = torch.tensor(test_loss).mean() / 500

    mean_losses.append([average_test_loss, lr])

    print(f"lr {lr}: {average_test_loss}")

lr 1.0: 13012.3447265625
lr 0.5: 4119.73291015625
lr 0.1: 4165.79345703125
lr 0.05: 2903.203857421875
lr 0.01: 525.6952514648438
lr 0.005: 43.011600494384766
lr 0.001: 168.1262664794922
lr 0.0005: 178.37498474121094
lr 0.0001: 186.249267578125


In [9]:
for loss, lr in sorted(mean_losses, key=lambda x: x[1]):
    print(f"lr {lr}: {loss}")

lr 0.0001: 186.249267578125
lr 0.0005: 178.37498474121094
lr 0.001: 168.1262664794922
lr 0.005: 43.011600494384766
lr 0.01: 525.6952514648438
lr 0.05: 2903.203857421875
lr 0.1: 4165.79345703125
lr 0.5: 4119.73291015625
lr 1.0: 13012.3447265625
